# 04 - Ensemble Best DL + Best ML

Notebook này ensemble xác suất của best ML model và một DL candidate mạnh nhất hiện tại.

- ML: lấy top model từ `models/ml/exported_top_models.csv`.
- DL: ưu tiên load `models/dl/best_dl_model.pt`; nếu chưa có thì train `FTTransformer|Weighted BCE` từ `train_data.csv`.
- Ensemble: `p = w * p_dl + (1 - w) * p_ml`.
- Tune trên validation: grid `w` và threshold để tối ưu `val_f1`.
- Đánh giá cuối trên test set và export metrics.

In [1]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as exc:
    raise ImportError('Notebook này cần PyTorch: pip install -r ../requirements.txt') from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cpu')

## 1. Load processed data

In [2]:
ROOT = Path('..')
TRAIN_PATH = ROOT / 'data/processed/train_data.csv'
VAL_PATH = ROOT / 'data/processed/validation_data.csv'
TEST_PATH = ROOT / 'data/processed/test_data.csv'
ML_EXPORT_PATH = ROOT / 'models/ml/exported_top_models.csv'
DL_EXPORT_PATH = ROOT / 'models/dl/exported_top_dl_models.csv'
DL_MODEL_PATH = ROOT / 'models/dl/best_dl_model.pt'
DL_META_PATH = ROOT / 'models/dl/best_dl_model_meta.json'

METRIC_DIR = ROOT / 'results/metrics'
MODEL_DIR = ROOT / 'models/ensemble'
METRIC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'depression_label'
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
feature_cols = [col for col in train_df.columns if col != TARGET]

def make_xy(df):
    return df[feature_cols].astype('float32').values, df[TARGET].astype(int).values

X_train, y_train = make_xy(train_df)
X_val, y_val = make_xy(val_df)
X_test, y_test = make_xy(test_df)

print('Train:', X_train.shape, pd.Series(y_train).value_counts().to_dict())
print('Validation:', X_val.shape, pd.Series(y_val).value_counts().to_dict())
print('Test:', X_test.shape, pd.Series(y_test).value_counts().to_dict())

Train: (2590, 18) {0: 2159, 1: 431}
Validation: (555, 18) {0: 463, 1: 92}
Test: (555, 18) {0: 463, 1: 92}


## 2. Helper metrics and threshold tuning

In [ ]:
def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'minority_recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
    }


def tune_threshold(y_true, y_prob):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 181):
        metrics = compute_metrics(y_true, y_prob, threshold)
        rows.append(metrics)
    df = pd.DataFrame(rows).sort_values(['f1', 'macro_f1', 'precision'], ascending=False)
    return float(df.iloc[0]['threshold']), df


def predict_proba_torch(model, X, batch_size=256):
    model.eval()
    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=batch_size)
    probs = []
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            logits = model(xb).squeeze(1)
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

## 3. Load best ML model

In [ ]:
def resolve_project_path(raw_path):
    raw_path = Path(str(raw_path))
    candidates = [
        raw_path,
        ROOT / str(raw_path).replace('../', ''),
        ROOT / raw_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy file artifact. Tried: {candidates}')


def unwrap_sklearn_artifact(artifact):
    if isinstance(artifact, dict):
        if 'model' not in artifact:
            raise KeyError("ML .pkl là dict nhưng không có key 'model'")
        return artifact['model'], artifact
    return artifact, {}


def predict_ml_proba(model, X_array, feature_cols):
    X_df = pd.DataFrame(X_array, columns=feature_cols)
    try:
        return model.predict_proba(X_df)[:, 1]
    except Exception:
        return model.predict_proba(X_array)[:, 1]


ml_export = pd.read_csv(ML_EXPORT_PATH).sort_values('val_f1', ascending=False)
best_ml_row = ml_export.iloc[0]
ml_path = resolve_project_path(best_ml_row['path'])

print(f"Best ML candidate: {best_ml_row['candidate_key']}")
print(f"Loading ML .pkl from: {ml_path}")

ml_artifact = joblib.load(ml_path)
ml_model, ml_meta = unwrap_sklearn_artifact(ml_artifact)
ml_feature_cols = ml_meta.get('feature_columns', feature_cols)

ml_val_prob = predict_ml_proba(ml_model, X_val, ml_feature_cols)
ml_test_prob = predict_ml_proba(ml_model, X_test, ml_feature_cols)

ml_threshold, ml_threshold_grid = tune_threshold(y_val, ml_val_prob)
ml_metrics = compute_metrics(y_val, ml_val_prob, ml_threshold)
ml_metrics.update({
    'candidate': best_ml_row['candidate_key'],
    'source': 'ML',
    'model_path': str(ml_path),
})

pd.DataFrame([ml_metrics])

AttributeError: 'dict' object has no attribute 'predict_proba'

## 4. DL candidate: load saved FTTransformer or train fallback

In [ ]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.feature_embedding = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.feature_embedding)

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0) + self.feature_embedding.unsqueeze(0)


class FTTransformerClassifier(nn.Module):
    def __init__(self, n_features, d_token=48, n_heads=4, n_layers=2, dropout=0.25):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1),
        )
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, x):
        tokens = self.tokenizer(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        encoded = self.encoder(torch.cat([cls, tokens], dim=1))
        return self.head(encoded[:, 0])

In [ ]:
def make_loader(X, y, batch_size=128, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_fttransformer_for_ensemble(lr=5e-4, epochs=140, patience=20):
    neg_count = int((y_train == 0).sum())
    pos_count = int((y_train == 1).sum())
    pos_weight = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32, device=DEVICE)
    model = FTTransformerClassifier(n_features=X_train.shape[1], d_token=48, n_heads=4, n_layers=2, dropout=0.25).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    train_loader = make_loader(X_train, y_train.astype('float32'), shuffle=True)
    val_loader = make_loader(X_val, y_val.astype('float32'))

    best_state = None
    best_val_f1 = -np.inf
    wait = 0
    rows = []
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(1), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            train_losses.append(float(loss.item()))

        val_prob = predict_proba_torch(model, X_val)
        threshold, _ = tune_threshold(y_val, val_prob)
        val_metrics = compute_metrics(y_val, val_prob, threshold)
        rows.append({'epoch': epoch, 'train_loss': np.mean(train_losses), 'val_f1': val_metrics['f1'], 'threshold': threshold})
        print(f"epoch={epoch:03d} train_loss={np.mean(train_losses):.4f} val_f1={val_metrics['f1']:.4f} threshold={threshold:.3f}")

        if val_metrics['f1'] > best_val_f1 + 1e-4:
            best_val_f1 = val_metrics['f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(rows)

In [ ]:
def get_best_dl_artifact_path():
    if DL_EXPORT_PATH.exists():
        dl_export = pd.read_csv(DL_EXPORT_PATH)
        dl_export = dl_export[dl_export['artifact_path'].notna()]
        if not dl_export.empty:
            return resolve_project_path(dl_export.iloc[0]['artifact_path'])
    if DL_MODEL_PATH.exists():
        return DL_MODEL_PATH
    return None


dl_artifact_path = get_best_dl_artifact_path()
if dl_artifact_path is not None and DL_META_PATH.exists():
    checkpoint = torch.load(dl_artifact_path, map_location=DEVICE)
    meta = json.loads(DL_META_PATH.read_text(encoding='utf-8'))
    dl_source = meta.get('model_name', 'saved_dl_model')
    if not str(dl_source).startswith('FTTransformer'):
        raise ValueError(f'Notebook ensemble hiện đang load FTTransformer .pt, nhưng artifact là: {dl_source}')

    dl_model = FTTransformerClassifier(
        n_features=len(meta.get('feature_cols', feature_cols)),
        d_token=48,
        n_heads=4,
        n_layers=2,
        dropout=0.25,
    ).to(DEVICE)
    state_dict = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
    dl_model.load_state_dict(state_dict)
    dl_model.eval()
    dl_train_log = pd.DataFrame()
    print(f'Best DL candidate: {dl_source}')
    print(f'Loading DL .pt from: {dl_artifact_path}')
else:
    print('Không tìm thấy DL .pt hợp lệ, train fallback FTTransformer|Weighted BCE.')
    dl_model, dl_train_log = train_fttransformer_for_ensemble(lr=5e-4, epochs=140, patience=20)
    dl_source = 'FTTransformer|Weighted BCE retrained'

dl_val_prob = predict_proba_torch(dl_model, X_val)
dl_test_prob = predict_proba_torch(dl_model, X_test)
dl_threshold, dl_threshold_grid = tune_threshold(y_val, dl_val_prob)
dl_metrics = compute_metrics(y_val, dl_val_prob, dl_threshold)
dl_metrics.update({'candidate': dl_source, 'source': 'DL'})
pd.DataFrame([dl_metrics])

## 5. Tune ensemble weight and threshold on validation

In [ ]:
ensemble_rows = []
for w_dl in np.linspace(0.0, 1.0, 41):
    val_prob = w_dl * dl_val_prob + (1 - w_dl) * ml_val_prob
    threshold, _ = tune_threshold(y_val, val_prob)
    metrics = compute_metrics(y_val, val_prob, threshold)
    metrics.update({'w_dl': w_dl, 'w_ml': 1 - w_dl})
    ensemble_rows.append(metrics)

ensemble_val_grid = pd.DataFrame(ensemble_rows).sort_values(['f1', 'macro_f1', 'precision'], ascending=False)
best_ensemble = ensemble_val_grid.iloc[0].to_dict()
ensemble_val_grid.to_csv(METRIC_DIR / 'ensemble_dl_ml_validation_grid.csv', index=False)
ensemble_val_grid.head(10)

## 6. Final test evaluation

In [ ]:
w_dl = best_ensemble['w_dl']
threshold = best_ensemble['threshold']
ensemble_test_prob = w_dl * dl_test_prob + (1 - w_dl) * ml_test_prob
test_metrics = compute_metrics(y_test, ensemble_test_prob, threshold)
test_metrics.update({
    'candidate': 'DL_ML_Probability_Ensemble',
    'dl_model': dl_source,
    'ml_model': best_ml_row['candidate_key'],
    'w_dl': w_dl,
    'w_ml': 1 - w_dl,
})

val_summary = pd.DataFrame([
    ml_metrics,
    dl_metrics,
    {**best_ensemble, 'candidate': 'DL_ML_Probability_Ensemble', 'source': 'Ensemble'},
])
test_summary = pd.DataFrame([test_metrics])

val_summary.to_csv(METRIC_DIR / 'ensemble_dl_ml_validation_metrics.csv', index=False)
test_summary.to_csv(METRIC_DIR / 'ensemble_dl_ml_test_metrics.csv', index=False)

test_pred = (ensemble_test_prob >= threshold).astype(int)
pd.DataFrame(confusion_matrix(y_test, test_pred), index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1']).to_csv(
    METRIC_DIR / 'ensemble_dl_ml_confusion_matrix.csv'
)
pd.DataFrame(classification_report(y_test, test_pred, output_dict=True, zero_division=0)).T.to_csv(
    METRIC_DIR / 'ensemble_dl_ml_classification_report.csv'
)

ensemble_meta = {
    'candidate': 'DL_ML_Probability_Ensemble',
    'dl_model': dl_source,
    'ml_model': str(best_ml_row['candidate_key']),
    'ml_path': str(ml_path),
    'w_dl': float(w_dl),
    'w_ml': float(1 - w_dl),
    'threshold': float(threshold),
    'feature_cols': feature_cols,
}
(MODEL_DIR / 'ensemble_meta.json').write_text(json.dumps(ensemble_meta, indent=2), encoding='utf-8')

display(val_summary)
display(test_summary)
ensemble_meta